In [31]:
import pandas as pd

# =============================
# Dataset Loading
# =============================

games_df = pd.read_csv("../Data/api data/Old data/Final Database/games.csv")
detailed_games_df = pd.read_csv("../Data/api data/Old data/Final Database/detailed_games.csv")
company_games_df = pd.read_csv("../Data/api data/Old data/Final Database/company_games.csv")
genre_games_df = pd.read_csv("../Data/api data/Old data/Final Database/genre_games.csv")
platform_df = pd.read_csv("../Data/api data/Old data/Final Database/platform.csv")


final_df = pd.read_csv("../Data/Dataset.csv")

C:\Users\cassy\AppData\Local\Temp\ipykernel_31568\1055050788.py:14: DtypeWarning:

Columns (8,25,33) have mixed types. Specify dtype option on import or set low_memory=False.



In [61]:
import dash
from dash import dcc, html, Input, Output, State, ctx, ALL
import dash_bootstrap_components as dbc
import threading
import webbrowser
import plotly.express as px
import plotly.graph_objects as go

# Create Dash app with Bootstrap theme
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
app.title = "Dashboard"

years = list(range(1970, 2025))

# =========================
# Helper component builders
# =========================

def build_top_control(active_tab):
    return dbc.ButtonGroup([
        dbc.Button("Games", id="games-button", n_clicks=0, color="primary" if active_tab == "games" else "secondary"),
        dbc.Button("Companies", id="companies-button", n_clicks=0, color="primary" if active_tab == "companies" else "secondary")
    ], id="top_control", style={"width": "100%"})

def build_search_bar():
    return dbc.Card(
        dbc.CardBody([
            dbc.Input(
                id="search_bar",
                placeholder="Search...",
                type="text",
                style={"marginBottom": "0"}
            )
        ]),
        style={
            "marginTop": "1rem",
            "marginBottom": "8rem",
            "boxShadow": "0px 2px 6px rgba(0,0,0,0.1)",
            "border": "1px solid #ced4da",
            "borderRadius": "0.5rem",
            "backgroundColor": "white"
        }
    )


def build_data_view():
    return html.Div(
        children=[
            html.H1("Main Visualization Area"),
            html.Div(
                html.Div(id="main_graph", style={"width": "100%"}),
                #dcc.Graph(id="main_graph", style={"height": "600px", "width": "100%"}),
                style={
                    "overflowX": "auto",
                    "width": "100%",
                    "paddingBottom": "1rem"
                }
            )
        ],
        id="data_view",
        style={"padding": "2rem"}
    )



def build_games_middle(selected_sub, selected_sort_options, selected_genres):
    # Left Column (always the same)
    buttons = []
    options = ["Most Popular", "Genres", "New Releases"]
    for label in options:
        idx = label.lower().replace(" ", "-") + "-sub"
        color = "primary" if selected_sub == idx else "secondary"
        buttons.append(
            dbc.Button(
                label,
                id={"type": "sub-button", "index": idx},
                color=color,
                outline=False,
                n_clicks=0,
                style={"width": "100%", "marginBottom": "0.5rem"}
            )
        )

    left_col = html.Div(buttons)

    # Right Column (dynamic depending on selected sub-button)
    if selected_sub == "most-popular-sub":
        sort_options = ["Rating", "YouTube", "Twitch", "Added", "Metacritic"]
        right_col = html.Div([
            html.H6("Sort By:"),
            *[
                dbc.Button(
                    m,
                    id={"type": "sort-button", "index": m},
                    color="primary" if m in selected_sort_options else "secondary",
                    outline=False,
                    style={"width": "100%", "marginBottom": "0.25rem"}
                )
                for m in sort_options
            ],
            html.Hr(),
            html.H6("Select Year Range:"),
            dcc.RangeSlider(
                id="year-range-slider",
                min=1970, max=2024, step=1,
                marks={str(y): str(y) for y in range(1970, 2025, 5)},
                value=[2000, 2020],
                vertical=True,
                verticalHeight=300
            )

        ], style={"paddingLeft": "1rem"})

    elif selected_sub == "genres-sub":
        genre_options = ["Action", "Shooter", "Farming"]
        right_col = html.Div([
            html.H6("Genres:"),
            *[
                dbc.Button(
                    g,
                    id={"type": "genre-button", "index": g},
                    color="primary" if g in selected_genres else "secondary",
                    outline=False,
                    style={"width": "100%", "marginBottom": "0.5rem"}
                )
                for g in genre_options
            ]
        ], style={"paddingLeft": "0.5rem"})

    else:
        right_col = html.Div()  # Empty for New Releases etc.

    return dbc.Row([dbc.Col(left_col, width=6), dbc.Col(right_col, width=6)])

def build_companies_middle(selected_sub):
    buttons = []
    options = ["Companies", "Publishers"]
    for label in options:
        idx = label.lower().replace(" ", "-") + "-sub"
        color = "primary" if selected_sub == idx else "secondary"
        buttons.append(
            dbc.Button(
                label,
                id={"type": "sub-button", "index": idx},
                color=color,
                outline=False,
                n_clicks=0,
                style={"width": "100%", "marginBottom": "0.5rem"}
            )
        )
    return html.Div(buttons)

def build_sidebar(active_tab, selected_sub, selected_sort_options, selected_genres):
    return html.Div([
        build_top_control(active_tab),
        html.Hr(),
        html.Div(
            id="middle_options",
            children=build_games_middle(selected_sub, selected_sort_options, selected_genres) if active_tab == "games" else build_companies_middle(selected_sub),
            style={"flexGrow": 1, "overflowY": "auto","overflowX": "hidden","paddingTop": "1rem","paddingBottom": "1rem"}
        ),
        html.Hr(),
        build_search_bar()
    ], style={
        "display": "flex",
        "flexDirection": "column",
        "height": "100%",
        "padding": "1rem"
    })


# =========================
# App Layout
# =========================

# Important: Default sidebar must be built immediately at startup!
initial_active_tab = "games"
initial_selected_sub = "most-popular-sub"

app.layout = dbc.Container(
    fluid=True,
    children=[
        dcc.Store(id="active_main_tab", data=initial_active_tab),
        dcc.Store(id="selected_sub_button", data=initial_selected_sub),
        dcc.Store(id="selected_sort_options", data=[]),
        dcc.Store(id="selected_genres", data=[]),
        dcc.Store(id="selected_year_range", data=[1970, 2024]),

        dbc.Row([
            # Sidebar
            dbc.Col(
                id="sidebar",
                children=build_sidebar(initial_active_tab, initial_selected_sub, [], []),
                width=3,
                style={
                    "backgroundColor": "#f8f9fa",
                    "height": "100vh",
                    "padding": 0,
                    "borderRight": "1px solid #dee2e6",
                    "display": "flex",
                    "flexDirection": "column"
                }
            ),

            # Data View
            dbc.Col(build_data_view(), width=9)
        ])
    ]
)

# =========================
# Callbacks
# =========================

@app.callback(
    [Output("active_main_tab", "data"),
     Output("selected_sub_button", "data")],
    [Input("games-button", "n_clicks"),
     Input("companies-button", "n_clicks"),
     Input({"type": "sub-button", "index": ALL}, "n_clicks")],
    [State("active_main_tab", "data"),
     State("selected_sub_button", "data")]
)
def handle_clicks(games_clicks, companies_clicks, sub_clicks, current_tab, selected_sub):
    triggered = ctx.triggered_id

    if triggered == "games-button":
        return "games", "most-popular-sub"
    elif triggered == "companies-button":
        return "companies", "companies-sub"
    elif isinstance(triggered, dict) and triggered.get("type") == "sub-button":
        return current_tab, triggered["index"]
    else:
        return current_tab, selected_sub

@app.callback(
    Output("sidebar", "children"),
    [Input("active_main_tab", "data"),
     Input("selected_sub_button", "data"),
     Input("selected_sort_options", "data"),
     Input("selected_genres", "data")]
)
def update_sidebar(active_tab, selected_sub, selected_sort_options, selected_genres):
    return build_sidebar(active_tab, selected_sub, selected_sort_options, selected_genres)

@app.callback(
    Output("selected_genres", "data"),
    Input({"type": "genre-button", "index": ALL}, "n_clicks"),
    State("selected_genres", "data"),
    prevent_initial_call=True
)
def toggle_genre_selection(n_clicks_list, selected_genres):
    triggered = ctx.triggered_id
    if not triggered:
        return dash.no_update

    genre = triggered["index"]
    if genre in selected_genres:
        selected_genres.remove(genre)
    else:
        selected_genres.append(genre)

    return selected_genres

@app.callback(
    Output("selected_sort_options", "data"),
    Input({"type": "sort-button", "index": ALL}, "n_clicks"),
    State("selected_sort_options", "data"),
    prevent_initial_call=True
)
def select_sort_option(n_clicks_list, selected_sort_options):
    triggered = ctx.triggered_id
    if not triggered:
        return dash.no_update

    sort_option = triggered["index"]
    return [sort_option]

@app.callback(
    Output("selected_year_range", "data"),
    Input("year-range-slider", "value"),
    prevent_initial_call=True
)
def update_selected_year_range(year_range):
    return year_range


@app.callback(
    Output("main_graph", "children"),
    [Input("active_main_tab", "data"),
     Input("selected_sub_button", "data"),
     Input("selected_sort_options", "data"),
     Input("selected_year_range", "data")]
)
def update_main_graph(active_tab, selected_sub_button, selected_sort_options, selected_year_range):
    if active_tab == "companies" and selected_sub_button == "companies-sub":
        company_counts = company_games_df["company"].value_counts().reset_index()
        company_counts.columns = ["Company", "Number of Games"]

        company_counts = company_counts.sort_values("Number of Games", ascending=False)

        fig = px.bar(
            company_counts,
            x="Company",
            y="Number of Games",
            title="Top Companies by Number of Games",
            labels={"Company": "Company", "Number of Games": "Number of Games"},
        )

        fig.update_layout(
            xaxis_tickangle=-45,
            height=600,
            margin=dict(l=50, r=30, t=50, b=150),
            bargap=0.2,
        )

        return fig

    elif active_tab == "games" and selected_sub_button == "most-popular-sub":
        # Filter detailed_games_df
        #df = detailed_games_df.copy()
        df = final_df.copy()
        df = df.drop_duplicates()
        df = df[(df["release_year"] >= selected_year_range[0]) & (df["release_year"] <= selected_year_range[1])]

        # Make sure to map to correct column names if needed
        column_mapping = {
            "rating": "rating",
            "youtube": "youtube_count",
            "twitch": "twitch_count",
            "added": "added",
            "metacritic": "metacritic"
        }

        # If no sort option selected yet, default to rating
        if not selected_sort_options:
            sort_by = "rating"
        else:
            sort_by = column_mapping.get(selected_sort_options[0].lower(), "rating")  # because button text is like "YouTube"



        # Drop missing data
        df = df.dropna(subset=["name", sort_by])
        df = df.sort_values(by=sort_by, ascending=False).head(50)


        fig = go.Figure(go.Treemap(
            labels=df["name"],
            parents=[""] * len(df),  # Flat (no hierarchy)
            values=df[sort_by],  # Block size
            textinfo="label+value"
        ))

        fig.update_layout(
            xaxis_tickangle=-45,
            height=700,
            margin=dict(t=50, l=25, r=25, b=25),
            bargap=0.2,
        )

        fig.update_traces(
            tiling=dict(
                pad=5,         # space between tiles
            ),
            textfont=dict(size=12),  # make labels more readable
            textinfo="label+value"   # or just "label"
        )

        # Normalize size (between 80px and 150px for visual clarity)
        min_size, max_size = 40, 200
        min_val, max_val = df[sort_by].min(), df[sort_by].max()
        df["scaled_size"] = df[sort_by].apply(
            lambda v: min_size + (max_size - min_size) * ((v - min_val) / (max_val - min_val)) if max_val != min_val else 100
        )

        # Generate the grid
        grid = html.Div([
            html.Div([
    html.Img(
        src=game["background_image"],
        title=f"{game['name']} — {sort_by.capitalize()}: {game[sort_by]:.2f}",
        style={
            "width": "60%",
            "height": "60%",
            "borderRadius": "50%",
            "objectFit": "cover",
            "marginBottom": "4px",
            "boxShadow": "0 1px 4px rgba(0,0,0,0.3)"
        }
    ),
    html.Div(game["name"], style={
        "fontSize": "11px",
        "fontWeight": "bold",
        "overflow": "hidden",
        "textOverflow": "ellipsis",
        "whiteSpace": "nowrap"
    }),
    html.Div(f"{sort_by.capitalize()}: {game[sort_by]:.2f}", style={
        "fontSize": "10px",
        "color": "#444"
    })
], style={
    "width": f"{game['scaled_size']}px",
    "height": f"{game['scaled_size']}px",
    "backgroundColor": "#e0e0e0",  # light gray
    "borderRadius": "8px",
    "display": "inline-block",
    "margin": "6px",
    "textAlign": "center",
    "padding": "6px",
    "overflow": "hidden",
    "boxShadow": "0 2px 6px rgba(0,0,0,0.2)"
})

            for _, game in df.iterrows()
        ], style={
            "textAlign": "center",
            "padding": "1rem"
        })


        return grid

    else:
        return px.bar(title="Select a tab to view data")


# =========================
# Run server
# =========================

def open_browser():
    webbrowser.open_new("http://127.0.0.1:8050/")

if __name__ == "__main__":
    threading.Timer(1, open_browser).start()
    app.run(debug=True, use_reloader=False)

